# project_final_runner.ipynb

Teljes CXR projektfuttató notebook:

1. datasetek és szegmentációs variánsok előkészítése;
2. raw / lung_masked / lung_crop split ellenőrzés;
3. modellenként optimalizált training beállítások;
4. egyesített `comparison_df` létrehozása;
5. leaderboard, epoch-görbék, confusion matrix + ROC sorok;
6. Grad-CAM + saliency összehasonlítás.

A notebook úgy van felépítve, hogy **minden modellt külön paraméterezve** futtathass, majd a végén az eredményeket egy közös `comparison_df`-be fűzze.


In [ ]:
from pathlib import Path
import os

if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")

if os.path.exists("/content/CXR/.git"):
    %cd /content/CXR
    !git pull
else:
    %cd /content
    !git clone https://github.com/csambrus/CXR.git
    %cd /content/CXR

!mkdir -p /root/.kaggle
!cp kaggle.json /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json

#!pip install -r ./requirements.txt
!pip install kaggle kagglehub


In [ ]:
#%load_ext autoreload
#%autoreload 2

from pathlib import Path
import json
import random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from src.runtime import setup_tensorflow_runtime

from src.config import (
    SEGMENTATION_RAW_DIR,
    SEGMENTATION_DATA_DIR,
    SEGMENTATION_MODELS_DIR,
    LUNG_MASK_DIR,
    LUNG_MASKED_DIR,
    LUNG_CROP_DIR,
    ensure_dir,
    save_json,
)

from src.download_dataset import download_all_datasets
setup_tensorflow_runtime()

print("SEGMENTATION_RAW_DIR:", SEGMENTATION_RAW_DIR)
print("SEGMENTATION_DATA_DIR:", SEGMENTATION_DATA_DIR)

download_all_datasets()


In [ ]:
from src.lung_segmentation import prepare_segmentation_dataset, create_splits, verify_png_files

verify_png_files(SEGMENTATION_RAW_DIR / "images", "Images")
verify_png_files(SEGMENTATION_RAW_DIR / "masks", "Masks")
prepare_segmentation_dataset(overwrite = False)
create_splits()

In [ ]:
from src.lung_segmentation import train_segmentation, evaluate_segmentation, verify_png_files

verify_png_files(SEGMENTATION_DATA_DIR / "images", "Images")
verify_png_files(SEGMENTATION_DATA_DIR / "masks", "Masks")

train_segmentation(epochs=20)
evaluate_segmentation()

In [ ]:
from src.lung_segmentation import plot_training_history, plot_predictions

plot_training_history()
plot_predictions()

In [ ]:
from src.lung_segmentation import generate_classifier_variants

generate_classifier_variants()

In [ ]:
from src.dataloader import create_splits, print_split_summary
from src.config import RAW_DIR, SPLITS_DIR
from pathlib import Path

# Convert RAW_DIR and SPLITS_DIR to Path objects if they are strings
source_root_path = Path(RAW_DIR)
split_dir_path = Path(SPLITS_DIR)

splits = create_splits(
    source_root=source_root_path,
    split_dir=split_dir_path,
    overwrite=True,
)

print_split_summary(split_dir_path)

## 0. Modulok újratöltése és globális kapcsolók

Colabban fontos a modulok újratöltése, különösen akkor, ha a `.py` fájlokat közben módosítottad.

A fő kapcsolók:

- `RUN_FULL_RETRAIN=True`: minden konfigurált modellt újratanít;
- `RUN_SEGMENTATION_TRAINING=False`: szegmentációt nem tanít újra, csak ellenőrzi / használja;
- `RUN_EXPLAINABILITY=True`: Grad-CAM + saliency összehasonlítást is futtat.


In [ ]:
import importlib
import pandas as pd
from pathlib import Path

import src.train
import src.evaluate
import src.compare_models
import src.compare_explainability
import src.lung_segmentation

importlib.reload(src.train)
importlib.reload(src.evaluate)
importlib.reload(src.compare_models)
importlib.reload(src.compare_explainability)
importlib.reload(src.lung_segmentation)

from src.runtime import setup_tensorflow_runtime
from src.config import (
    RAW_DIR,
    LUNG_MASKED_DIR,
    LUNG_CROP_DIR,
    MODELS_DIR,
    SPLITS_DIR,
    OUTPUT_DIR,
    ensure_dir,
    save_json,
)
from src.dataloader import print_split_summary
from src.compare_models import (
    run_multiple_models,
    print_leaderboard,
    plot_all_main_metrics,
    plot_all_training_histories,
    plot_epoch_comparisons,
)
from src.compare_explainability import run_compare_explainability

setup_tensorflow_runtime()

RUN_FULL_RETRAIN = True          # teljes újrafuttatáshoz True
RUN_SEGMENTATION_TRAINING = False # ha a lung_unet már jó, maradjon False
RUN_EXPLAINABILITY = True

DATA_VARIANTS = ["raw", "lung_masked", "lung_crop"]
FINAL_COMPARISON_NAME = "project_final_optimized"
FINAL_OUT_DIR = ensure_dir(Path(MODELS_DIR) / FINAL_COMPARISON_NAME)

print("RAW_DIR        :", RAW_DIR)
print("LUNG_MASKED_DIR:", LUNG_MASKED_DIR)
print("LUNG_CROP_DIR  :", LUNG_CROP_DIR)
print("MODELS_DIR     :", MODELS_DIR)
print("SPLITS_DIR     :", SPLITS_DIR)
print("OUTPUT_DIR     :", OUTPUT_DIR)
print("FINAL_OUT_DIR  :", FINAL_OUT_DIR)


## 1. Split ellenőrzés

A raw / lung_masked / lung_crop variánsok ugyanazt a splitet használják. Ez azért jó, mert így a modellek és variánsok összehasonlítása korrekt: ugyanazok a képek kerülnek train / val / test halmazba.


In [ ]:
print_split_summary(SPLITS_DIR)


## 2. EfficientNetB0 sanity check

Ez a cella ellenőrzi, hogy a javított `train.py` tényleg bekerült-e a futó notebookba. EfficientNetB0 esetén kell lennie `scale_to_255` rétegnek. Ha nincs, akkor a régi `train.py` fut, és az EfficientNet várhatóan rosszul fog szerepelni.


In [ ]:
from src.train import build_model

_tmp_model, _tmp_base = build_model("efficientnetb0", pretrained=True)
layer_names = [layer.name for layer in _tmp_model.layers]
print("EfficientNet model layers:", layer_names)

if "scale_to_255" not in layer_names:
    raise RuntimeError(
        "[ERROR] EfficientNetB0 modelben nincs scale_to_255 réteg. "
        "Frissítsd / reloadold a javított train.py-t!"
    )
else:
    print("[OK] EfficientNetB0 scale_to_255 réteg megtalálva.")


## 3. Modellenként optimalizált konfigurációk

Itt minden architektúra külön paramétereket kap. Ez jobb, mint ugyanazt a learning rate / epoch beállítást ráerőltetni minden modellre.


In [ ]:
MODEL_RUN_CONFIGS = [
    {
        "label": "resnet50_optimized",
        "model_names": ["resnet50"],
        "data_variants": DATA_VARIANTS,
        "pretrained": True,
        "do_fine_tuning": True,
        "epochs_head": 8,
        "epochs_finetune": 6,
        "learning_rate_head": 1e-3,
        "learning_rate_finetune": 1e-5,
        "comparison_name": "resnet50_optimized",
    },
    {
        "label": "vgg16_optimized",
        "model_names": ["vgg16"],
        "data_variants": DATA_VARIANTS,
        "pretrained": True,
        "do_fine_tuning": True,
        "epochs_head": 10,
        "epochs_finetune": 6,
        "learning_rate_head": 5e-4,
        "learning_rate_finetune": 1e-5,
        "comparison_name": "vgg16_optimized",
    },
    {
        "label": "efficientnetb0_fixed_optimized",
        "model_names": ["efficientnetb0"],
        "data_variants": DATA_VARIANTS,
        "pretrained": True,
        "do_fine_tuning": True,
        "epochs_head": 12,
        "epochs_finetune": 12,
        "learning_rate_head": 3e-4,
        "learning_rate_finetune": 3e-6,
        "comparison_name": "efficientnetb0_fixed_optimized",
    },
    {
        "label": "baseline_cnn_optimized",
        "model_names": ["baseline_cnn"],
        "data_variants": DATA_VARIANTS,
        "pretrained": False,
        "do_fine_tuning": False,
        "epochs_head": 18,
        "epochs_finetune": 0,
        "learning_rate_head": 1e-3,
        "learning_rate_finetune": 1e-5,
        "comparison_name": "baseline_cnn_optimized",
    },
]

MODEL_RUN_CONFIGS


## 4. Modellek futtatása külön konfigurációkkal

Ez a cella minden konfigurációt külön futtat, majd a kapott dataframe-eket eltárolja a `model_result_dfs` listába.

Teljes újrafuttatáshoz `RUN_FULL_RETRAIN=True`, ilyenkor `skip_if_complete=False`. Ha csak ellenőrizni akarod a már kész eredményeket, állítsd `RUN_FULL_RETRAIN=False`-ra.


In [ ]:
model_result_dfs = []

for cfg in MODEL_RUN_CONFIGS:
    print("
" + "=" * 90)
    print("RUN CONFIG:", cfg["label"])
    print("=" * 90)

    df = run_multiple_models(
        split_dir=SPLITS_DIR,
        out_dir=MODELS_DIR,
        model_names=cfg["model_names"],
        data_variants=cfg["data_variants"],
        pretrained=cfg["pretrained"],
        do_fine_tuning=cfg["do_fine_tuning"],
        epochs_head=cfg["epochs_head"],
        epochs_finetune=cfg["epochs_finetune"],
        learning_rate_head=cfg["learning_rate_head"],
        learning_rate_finetune=cfg["learning_rate_finetune"],
        comparison_name=cfg["comparison_name"],
        make_plots=True,
        show_plots=True,
        skip_if_complete=not RUN_FULL_RETRAIN,
        trust_existing_without_fingerprint=True,
    )

    df["run_config"] = cfg["label"]
    model_result_dfs.append(df)

print("[OK] Egyedi modellfuttatások kész.")


## 5. Egyesített comparison_df létrehozása

Itt jön létre a végső `comparison_df`. Ha ugyanaz a modell/variáns több futásból is bekerülne, az utolsó eredményt tartjuk meg.


In [ ]:
comparison_df = pd.concat(model_result_dfs, ignore_index=True)

# Biztonsági deduplikálás: model + data_variant alapján az utolsó futás marad.
comparison_df = (
    comparison_df
    .drop_duplicates(subset=["model", "data_variant"], keep="last")
    .sort_values(["f1_macro", "accuracy", "recall_macro"], ascending=False)
    .reset_index(drop=True)
)

comparison_csv = Path(FINAL_OUT_DIR) / "comparison.csv"
leaderboard_csv = Path(FINAL_OUT_DIR) / "leaderboard.csv"
comparison_json = Path(FINAL_OUT_DIR) / "comparison.json"

comparison_df.to_csv(comparison_csv, index=False)
comparison_df.to_csv(leaderboard_csv, index=False)
save_json({"rows": comparison_df.to_dict(orient="records")}, comparison_json)

print("[INFO] Saved:", comparison_csv)
print("[INFO] Saved:", leaderboard_csv)
print("[INFO] Saved:", comparison_json)

print_leaderboard(comparison_df)
display(comparison_df)


## 6. Végső összehasonlító ábrák

A közös `comparison_df` alapján újra elkészítjük a végső bar chartokat, heatmapeket és epoch összehasonlításokat.


In [ ]:
plot_all_main_metrics(comparison_df, FINAL_OUT_DIR, show=True)
plot_all_training_histories(comparison_df, FINAL_OUT_DIR, show=True)
plot_epoch_comparisons(comparison_df, FINAL_OUT_DIR, show=True)

print("[OK] Final comparison plots saved to:", FINAL_OUT_DIR)


## 7. Grad-CAM + Saliency összehasonlítás

Ez a végső modellekre fut. Ha csak a legjobb modelleket akarod magyarázni, használd a `best_models` cellát; ha minden fő transfer modellt össze akarsz hasonlítani, hagyd a megadott listát.


In [ ]:
if RUN_EXPLAINABILITY:
    explainability_summary = run_compare_explainability(
        model_names=["resnet50", "vgg16", "efficientnetb0"],
        data_variants=DATA_VARIANTS,
        split_dir=SPLITS_DIR,
        out_dir=Path(OUTPUT_DIR) / "figures" / "project_final_explainability",
        n_examples=6,
        include_saliency=True,
        show=True,
        skip_existing=False,
    )
    explainability_summary
else:
    print("[SKIP] Explainability disabled.")


## 8. Legjobb modell variánsonként

Ez hasznos a jelentéshez: megmutatja, hogy raw / lung_masked / lung_crop esetén melyik modell volt a legjobb macro-F1 alapján.


In [ ]:
best_by_variant = (
    comparison_df
    .sort_values("f1_macro", ascending=False)
    .groupby("data_variant", as_index=False)
    .first()
)

best_by_model = (
    comparison_df
    .sort_values("f1_macro", ascending=False)
    .groupby("model", as_index=False)
    .first()
)

best_by_variant.to_csv(Path(FINAL_OUT_DIR) / "best_by_variant.csv", index=False)
best_by_model.to_csv(Path(FINAL_OUT_DIR) / "best_by_model.csv", index=False)

print("Best by variant:")
display(best_by_variant)

print("Best by model:")
display(best_by_model)


## 9. Notebook mentése Drive-ra

Colabban mentési kérést küld a notebook aktuális állapotára.


In [ ]:
try:
    from google.colab import _message
    _message.blocking_request('request_save', timeout_sec=10)
    print("Notebook save requested.")
except Exception as e:
    print("[WARN] Notebook save request failed or not running in Colab:", e)
